# Tutorial Completo: Transformaciones y Normalización de Datos

En este cuaderno aprenderemos las técnicas fundamentales para transformar y normalizar datos antes de entrenar modelos de Machine Learning. Veremos:
1. Manejo de asimetría (Skewness) con transformaciones logarítmicas y Box-Cox.
2. Normalización Min-Max (MinMaxScaler).
3. Estandarización (StandardScaler).
4. Transformación Robusta (RobustScaler).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, PowerTransformer

# Configuración de estilo
sns.set_theme(style="whitegrid")

## 1. Generación de un Dataset Sintético

Crearemos un conjunto de datos que simule variables con diferentes escalas y distribuciones sesgadas.

In [ ]:
np.random.seed(42)
n = 1000

# Variable con sesgo positivo (exponencial)
income = np.random.exponential(scale=20000, size=n) + 15000

# Variable con distribución normal estándar
age = np.random.normal(loc=35, scale=10, size=n)

# Variable con outliers
score = np.random.normal(loc=500, scale=50, size=n)
score[0:10] = score[0:10] * 3  # Añadir outliers extremos

df = pd.DataFrame({
    'Income': income,
    'Age': age,
    'Score': score
})

df.head()

Visualicemos las distribuciones originales de nuestros datos.

In [ ]:
plt.figure(figsize=(15, 4))
for i, col in enumerate(df.columns):
    plt.subplot(1, 3, i + 1)
    sns.histplot(df[col], kde=True, color='skyblue')
    plt.title(f'Distribución Original: {col}')
plt.tight_layout()
plt.show()

## 2. Corrección de Asimetría (Skewness)

La variable 'Income' presenta un claro sesgo positivo. Aplicaremos una transformación logarítmica y una transformación de Box-Cox para hacerla más simétrica.

In [ ]:
df['Income_Log'] = np.log1p(df['Income'])

pt = PowerTransformer(method='box-cox')
df['Income_BoxCox'] = pt.fit_transform(df[['Income']])

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
sns.histplot(df['Income'], kde=True, color='red')
plt.title('Original Income')

plt.subplot(1, 3, 2)
sns.histplot(df['Income_Log'], kde=True, color='green')
plt.title('Log Transform')

plt.subplot(1, 3, 3)
sns.histplot(df['Income_BoxCox'], kde=True, color='purple')
plt.title('Box-Cox Transform')

plt.tight_layout()
plt.show()

## 3. Escalado y Normalización

### A. MinMaxScaler (Normalización)
Escala los datos a un rango específico, comúnmente entre 0 y 1.

In [ ]:
min_max_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(
    min_max_scaler.fit_transform(df[['Age', 'Score']]),
    columns=['Age_MinMax', 'Score_MinMax']
)
df_minmax.describe()

### B. StandardScaler (Estandarización)
Centra los datos restando la media y dividiendo por la desviación estándar (media = 0, desviación estándar = 1).

In [ ]:
standard_scaler = StandardScaler()
df_std = pd.DataFrame(
    standard_scaler.fit_transform(df[['Age', 'Score']]),
    columns=['Age_Std', 'Score_Std']
)
df_std.describe()

### C. RobustScaler
Utiliza la mediana y el rango intercuartílico, lo que lo hace muy robusto ante la presencia de valores atípicos (outliers).

In [ ]:
robust_scaler = RobustScaler()
df_robust = pd.DataFrame(
    robust_scaler.fit_transform(df[['Age', 'Score']]),
    columns=['Age_Robust', 'Score_Robust']
)
df_robust.describe()

## 4. Conclusión

- Utiliza **Log o Box-Cox** para corregir asimetrías antes de modelos lineales.
- Utiliza **StandardScaler** por defecto para la mayoría de algoritmos basados en gradiente o distancia (como SVM, KNN, Redes Neuronales).
- Utiliza **MinMaxScaler** si requieres estrictamente un rango acotado (como en redes neuronales de activación sigmoide).
- Utiliza **RobustScaler** cuando tu dataset contenga muchos outliers que no deseas eliminar.